# Graph deduplication / cleanup - agent approach

In [ ]:
import json
import os
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"
import re
from typing import List, Dict, Optional
from neo4j import GraphDatabase
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from pydantic import BaseModel


# LLM_MODEL_NAME="SpeakLeash/bielik-11b-v2.3-instruct:Q8_0"
LLM_MODEL_NAME = "gpt-oss:20b"
DENSE_MODEL_NAME = "sdadas/stella-pl-retrieval"
RERANK_MODEL_NAME = "sdadas/polish-reranker-roberta-v3"
NEO4J_SPARSE_INDEX="node_fulltext_index"
NEO4J_DENSE_INDEX="node_embedding_index"
NEO4J_REL_DENSE_INDEX="rel_embedding_index"
NEO4J_REL_SPARSE_INDEX="rel_fulltext_index"
NEO4J_URL = "neo4j://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

class Relation(BaseModel):
    head: str
    head_type: str
    relation: str
    tail: str
    tail_type: str

DENSE_MODEL = HuggingFaceEmbeddings(
    model=DENSE_MODEL_NAME,
    model_kwargs={"trust_remote_code": True, "device": "cpu"},
)

RERANKER = HuggingFaceCrossEncoder(
    model_name=RERANK_MODEL_NAME,
    model_kwargs={'trust_remote_code': True, 'device': 'cpu'}
)

## Logic

Code for the langgraph agent logic, the agent will be run when adding a subgraph to the main graph (the graph from a given chunk).

## 1. List of actions the langgraph agent will be able to perform:
```
## read_graph_schema

Description: Fetches the list of available Labels (node types) and Relationship Types.

Purpose: Keeping naming consistent (avoiding synonyms in types).

____________________________________________________________________________________________________________________________________

## search_neighborhood *(modified "Search")*

Description: Searches for nodes (vector/fulltext) and returns their direct neighborhood (1-hop).

Purpose: Understanding the context before any change.

____________________________________________________________________________________________________________________________________

## inspect_node_details *(New)*

Description: Returns all properties of a specific node (by ID).

Purpose: Verifying details before merging/deleting (e.g. checking dates, URLs).

____________________________________________________________________________________________________________________________________

## merge_nodes

Description: Merges two or more nodes into one (keeping all relations).

Purpose: Entity deduplication (e.g. "USA" and "United States").

____________________________________________________________________________________________________________________________________

## deduplicate_relations *(Modified "Merging relations")*

Description: Removes duplicated relations of the same type between the same two nodes.

Purpose: Cleaning up noise after processing the same documents multiple times.

____________________________________________________________________________________________________________________________________

## add_triplet *(Modified "Adding node/rel")*

Description: Runs a `MERGE` operation for the structure  
`(StartNode) -[REL]-> (EndNode)`.  
Creates nodes only if they do not exist.

Purpose: Adding new knowledge in a safe way.

____________________________________________________________________________________________________________________________________

## update_node_properties

Description: Changes the name or type, or adds metadata to a node.

Purpose: Fixing typos, unifying names (normalization).

____________________________________________________________________________________________________________________________________

## split_node_and_move_relations *(Modified "Splitting a node")*

Description: Creates a new node and moves selected relations to it from the faulty "super-node".

Purpose: Fixing errors where two people with the same surname were glued into one node.

____________________________________________________________________________________________________________________________________

## redirect_relation *(Modified "Splitting a relation")*

Description: Changes the target of an existing relation  
(replugs the cable from node A to node B).

Purpose: Fixing wrong links.

____________________________________________________________________________________________________________________________________

## delete_element *(Combined node/rel deletion)*

Description: Deletes the given node (with its relations) OR the given relation.

Purpose: Removing hallucinations or junk data.

____________________________________________________________________________________________________________________________________

## do_nothing

Description: Finish signal, when the graph is consistent.
```

## 2. Methods in the GraphRepository class matching the actions above.

## 3. Modify the langgraph sample code below so that it deduplicates / cleans the graph stored in neo4j. Use GraphRepository - find_nearest_nodes and find_nearest_relations. The agent methods must use the GraphRepository methods.

## 4. Agent logic when adding a subgraph to the main graph:
    - A subgraph is to be inserted as a set of triples (objects of the Relation class). Such a subgraph from a chunk is inserted into the main graph in neo4j.
    - Start by finding in the main graph the nodes nearest to the nodes of the chunk subgraph being inserted (find_nearest_nodes).
    - For the nodes found in the main graph, find all the nodes 'surrounding' them (connected by a relation) - the _expand_subgraph method and the hops logic. This gives us subgraphs from the main graph.
    - The subgraphs obtained this way from the main graph are compared with the chunk subgraph. The llm decides to perform one of the actions from point 1, e.g.:
        - when there are no similar nodes in the main-graph subgraphs relative to the chunk subgraph, we perform the "no action / do nothing" action
        - when the main-graph subgraphs contain the node "Putin" and the chunk subgraph has "Vladimir Putin", the main-graph node should be unified with the "updating nodes" action before adding nodes and relations to the main graph


In [ ]:
class GraphRepository:
    def __init__(self):
        self.driver = GraphDatabase.driver(NEO4J_URL, auth=(NEO4J_USER, NEO4J_PASSWORD))
        self.dense = DENSE_MODEL

    def _embed_text(self, text: str) -> list[float]:
        return self.dense.embed_query(text)

    def _embed_node(self, name: str, type_: str) -> list[float]:
        return self._embed_text(f"{type_}: {name}")

    def close(self):
        self.driver.close()

    @staticmethod
    def escape_lucene(text: str) -> str:
        return re.sub(r'([+\-!(){}[\]^"~*?:\\/])', r'\\\1', text)

    def update_chunk_graph(self, chunk_id: str, subgraph: List[Relation]):
        """
        Updates Neo4j with a "sub"-graph (the graph for a specific chunk).
        Each node has a type and a chunk_id list, relations between nodes are created by LM-service.
        """
        with self.driver.session() as session:
            for rel in subgraph:
                head_emb_calc = self._embed_node(rel.head, rel.head_type)
                tail_emb_calc = self._embed_node(rel.tail, rel.tail_type)
                rel_emb_calc = self._embed_text(rel.relation)
                session.execute_write(self._merge_relation, chunk_id, rel,
                                      head_emb_calc,
                                      tail_emb_calc,
                                      rel_emb_calc)

    def find_nearest_nodes(self, nodes: List[Dict[str, str]], top_k: int = 8) -> Dict[str, List[Dict]]:
        results = {}
        initial_k = top_k * 5
        hybrid_query = """
        CALL db.index.vector.queryNodes($vector_index, $k, $embedding)
        YIELD node, score
        WITH collect({node: node, score: score}) AS v

        CALL db.index.fulltext.queryNodes($fulltext_index, $text_query, {limit: $k})
        YIELD node, score
        WITH v, collect({node: node, score: score}) AS f

        WITH v + f AS allResults
        UNWIND allResults AS r
        WITH r.node AS n, max(r.score) AS finalScore
        RETURN elementId(n) AS id, n.name AS name, n.type AS type, finalScore AS score
        ORDER BY score DESC LIMIT $k
        """
        with self.driver.session() as session:
            for node in nodes:
                emb = self._embed_node(node['name'], node['type'])
                query_string = f"{node['type']}: {node['name']}"
                tokens = query_string.replace(":", "").split()
                text_query = " ".join([f"{self.escape_lucene(t)}~" for t in tokens])

                raw_res = session.run(
                    hybrid_query,
                    vector_index=NEO4J_DENSE_INDEX,
                    fulltext_index=NEO4J_SPARSE_INDEX,
                    k=initial_k,
                    embedding=emb,
                    text_query=text_query
                ).data()

                if not raw_res:
                    results[query_string] = []
                    continue

                pairs = [[query_string, f"{r['type']}: {r['name']}"] for r in raw_res]
                scores = RERANKER.score(pairs)
                for i, score in enumerate(scores):
                    raw_res[i]['rerank_score'] = float(score)

                final_nodes = sorted(raw_res, key=lambda x: x['rerank_score'], reverse=True)[:top_k]
                results[query_string] = final_nodes

        return results

    def find_nearest_relations(self, relations: List[str], top_k: int = 8) -> Dict[str, List[Dict]]:
        results = {}
        initial_k = top_k * 5
        hybrid_query = """
        CALL db.index.vector.queryRelationships($vector_index, $k, $embedding)
        YIELD relationship, score
        WITH collect({rel: relationship, score: score}) AS v

        CALL db.index.fulltext.queryRelationships($fulltext_index, $text_query, {limit: $k})
        YIELD relationship, score
        WITH v, collect({rel: relationship, score: score}) AS f

        WITH v + f AS allResults
        UNWIND allResults AS r
        WITH r.rel AS rel, max(r.score) AS finalScore
        RETURN elementId(startNode(rel)) AS head_id,
               startNode(rel).name AS head,
               startNode(rel).type AS head_type,
               rel.type AS relation_name,
               elementId(rel) AS relation_id,
               elementId(endNode(rel)) AS tail_id,
               endNode(rel).name AS tail,
               endNode(rel).type AS tail_type,
               finalScore AS score
        ORDER BY score DESC LIMIT $k
        """

        with self.driver.session() as session:
            for query_text in relations:
                emb = self._embed_text(query_text)
                tokens = query_text.split()
                text_query = " ".join([f"{self.escape_lucene(t)}~" for t in tokens])

                raw_res = session.run(
                    hybrid_query,
                    vector_index=NEO4J_REL_DENSE_INDEX,
                    fulltext_index=NEO4J_REL_SPARSE_INDEX,
                    k=initial_k,
                    embedding=emb,
                    text_query=text_query
                ).data()

                if not raw_res:
                    results[query_text] = []
                    continue

                pairs = []
                for r in raw_res:
                    doc_text = f"{r['head']} ({r['head_type']}) - {r['relation_name']} -> {r['tail']} ({r['tail_type']})"
                    pairs.append([query_text, doc_text])

                scores = RERANKER.score(pairs)
                for i, score in enumerate(scores):
                    raw_res[i]['rerank_score'] = float(score)

                final_rels = sorted(raw_res, key=lambda x: x['rerank_score'], reverse=True)[:top_k]
                results[query_text] = final_rels

        return results

    def _expand_subgraph(self, start_node_ids: list, hops: int, limit: int) -> List[str]:
        """
        Expands the context by 'hops' hops from the given start nodes.
        Returns node and relation IDs in the text so the Agent can use them.
        """
        if not start_node_ids or hops < 1:
            return []

        # 'Traversing' query
        # DISTINCT r, so relations are not duplicated if the graph has cycles
        cypher = f"""
            MATCH (start)
            WHERE elementId(start) IN $ids
            MATCH path = (start)-[r*1..{hops}]-(end)
            UNWIND relationships(path) AS rel
            WITH DISTINCT rel
            RETURN elementId(startNode(rel)) as head_id,
                   startNode(rel).name AS head,
                   startNode(rel).type AS head_type,
                   elementId(rel) as rel_id,
                   rel.type AS relation,
                   elementId(endNode(rel)) as tail_id,
                   endNode(rel).name AS tail,
                   endNode(rel).type AS tail_type
            LIMIT $limit
        """

        expanded_lines = []
        with self.driver.session() as session:
            results = session.run(cypher, ids=start_node_ids, limit=limit).data()

            for res in results:
                line = (f"[ID: {res['head_id']}] ({res['head_type']}: {res['head']}) "
                        f"--[ID: {res['rel_id']} {res['relation']}]--> "
                        f"[ID: {res['tail_id']}] ({res['tail_type']}: {res['tail']})")
                expanded_lines.append(line)

        return expanded_lines

    def build_nearest_subgraphs(self, nearest_nodes: dict, nearest_relations: list,
                                max_triples: int = 5, max_nodes: int = 8, hops: int = 2) -> str:

        context_parts = []
        collected_ids = set()

        # Relations handling
        if nearest_relations:
            sorted_rels = sorted(
                nearest_relations,
                key=lambda r: r.get("rerank_score", 0),
                reverse=True
            )[:max_triples]

            lines = []
            for idx, rel in enumerate(sorted_rels):
                if 'head_id' in rel: collected_ids.add(rel['head_id'])
                if 'tail_id' in rel: collected_ids.add(rel['tail_id'])

                head = rel.get("head", "UNKNOWN")
                head_type = rel.get("head_type", "Node")
                head_id = rel.get("head_id", "UNKNOWN")

                tail = rel.get("tail", "UNKNOWN")
                tail_type = rel.get("tail_type", "Node")
                tail_id = rel.get("tail_id", "UNKNOWN")

                rel_name = rel.get("relation_name", rel.get("relation", "UNKNOWN"))
                rel_id = rel.get("relation_id", "UNKNOWN")

                lines.append(
                    f"Znaleziona Relacja {idx}: "
                    f"[ID: {head_id}] ({head_type}: {head}) --[ID: {rel_id} {rel_name}]--> [ID: {tail_id}] ({tail_type}: {tail})"
                )

            if lines:
                context_parts.append("BEZPOŚREDNIE TRAFIENIA (RELACJE):\n" + "\n".join(lines))

        # Nodes handling
        if nearest_nodes:
            all_nodes = []
            for _, items in nearest_nodes.items():
                all_nodes.extend(items)

            # Deduplication by name and type
            seen = set()
            unique_nodes = []
            for n in all_nodes:
                nid = n.get('id')
                if nid and nid not in seen:
                    seen.add(nid)
                    unique_nodes.append(n)
                    collected_ids.add(nid)

            unique_nodes = sorted(unique_nodes, key=lambda n: n.get("rerank_score", 0), reverse=True)[:max_nodes]

            lines = []
            for idx, n in enumerate(unique_nodes):
                lines.append(f"Znaleziony Byt {idx}: [ID: {n.get('id')}] {n.get('name')} (Typ: {n.get('type')})")

            if lines:
                context_parts.append("BEZPOŚREDNIE TRAFIENIA (BYTY):\n" + "\n".join(lines))

        # 'hop' logic (Graph Expansion)
        if hops > 0 and collected_ids:
            neighborhood = self._expand_subgraph(list(collected_ids), hops=hops, limit=30)

            if neighborhood:
                context_parts.append(
                    f"\nSZERSZY KONTEKST (Sąsiedztwo {hops}-hop):\n" +
                    "Legenda: [ID: <node_id>] (<Type>: <Name>) --[ID: <rel_id> <RelType>]--> ...\n" +
                    "\n".join(neighborhood)
                )

        if not context_parts:
            return ""

        return "\n\n".join(context_parts)

    @staticmethod
    def _merge_relation(tx, chunk_id: str, rel: Relation, head_emb: list[float], tail_emb: list[float], rel_emb: list[float]):
        """
        1. Creates or updates (upsert) head and tail (adds chunk_id to the list if missing)
        2. Creates the relation between them
        """
        # MERGE head node
        tx.run(
            """
            MERGE (h:Node {name: $head_name, type: $head_type})
            ON CREATE SET
                h.chunk_ids = [$chunk_id],
                h.embedding = $embedding,
                h.text = $text
            ON MATCH SET h.chunk_ids = CASE WHEN NOT $chunk_id IN h.chunk_ids THEN h.chunk_ids + $chunk_id ELSE h.chunk_ids END
            """,
            head_name=rel.head,
            head_type=rel.head_type,
            chunk_id=chunk_id,
            embedding=head_emb,
            text=f"{rel.head_type}: {rel.head}"
        )

        # MERGE tail node
        tx.run(
            """
            MERGE (t:Node {name: $tail_name, type: $tail_type})
            ON CREATE SET
                t.chunk_ids = [$chunk_id],
                t.embedding = $embedding,
                t.text = $text
            ON MATCH SET t.chunk_ids = CASE WHEN NOT $chunk_id IN t.chunk_ids THEN t.chunk_ids + $chunk_id ELSE t.chunk_ids END
            """,
            tail_name=rel.tail,
            tail_type=rel.tail_type,
            chunk_id=chunk_id,
            embedding=tail_emb,
            text=f"{rel.tail_type}: {rel.tail}"
        )

        # MERGE relation
        tx.run(
            """
            MATCH (h:Node {name: $head_name, type: $head_type})
            MATCH (t:Node {name: $tail_name, type: $tail_type})
            MERGE (h)-[r:RELATED]->(t)
            ON CREATE SET
                r.chunk_ids = [$chunk_id],
                r.type = $original_rel_type,
                r.embedding = $rel_embedding
            ON MATCH SET r.chunk_ids = CASE WHEN NOT $chunk_id IN r.chunk_ids THEN r.chunk_ids + $chunk_id ELSE r.chunk_ids END
            """,
            head_name=rel.head,
            head_type=rel.head_type,
            tail_name=rel.tail,
            tail_type=rel.tail_type,
            chunk_id=chunk_id,
            original_rel_type=rel.relation,
            rel_embedding=rel_emb
        )

    # --- METHODS FOR THE AGENT (TOOLS IMPLEMENTATION) ---

    def get_schema_summary(self) -> str:
        """
        [Action: read_graph_schema]
        Fetches unique node types (property 'type') and relation types.
        Helps the agent understand the graph ontology.
        """
        query = """
        CALL () {
            MATCH (n:Node) RETURN collect(DISTINCT n.type) as nodeTypes
        }
        CALL () {
            MATCH ()-[r]->() RETURN collect(DISTINCT r.type) as relTypes
        }
        RETURN nodeTypes, relTypes
        """
        with self.driver.session() as session:
            res = session.run(query).single()
            if res:
                return f"Node Types: {res['nodeTypes']}\nRelation Types: {res['relTypes']}"
            return "Schema is empty."

    def search_neighborhood(self, query: str, top_k: int = 3, hops: int = 1) -> str:
        """
        [Action: search_neighborhood]
        1. Looks for the nearest nodes for the query (Hybrid Search).
        2. Fetches their direct neighborhood (1-hop or more).
        3. Returns formatted text with a list of relations so the agent can 'see' the graph.
        """
        # 1. Find the entry points (nodes)
        # Prepare the list in the format expected by find_nearest_nodes
        nodes_to_search = [{"name": query, "type": "Node"}]
        nearest_nodes_map = self.find_nearest_nodes(nodes_to_search, top_k)

        # 2. Extract the IDs of the found nodes
        collected_ids = []
        for nodes_list in nearest_nodes_map.values():
            for node in nodes_list:
                if 'id' in node:
                    collected_ids.append(node['id'])

        if not collected_ids:
            return f"Nie znaleziono węzłów pasujących do zapytania: '{query}'"

        # 3. Fetch the neighborhood (Traversing)
        neighborhood_triples = self._expand_subgraph(collected_ids, hops=hops, limit=20)

        if not neighborhood_triples:
            # If we found nodes but they have no relations
            found_names = [n['name'] for n_list in nearest_nodes_map.values() for n in n_list]
            return f"Znaleziono węzły: {found_names}, ale nie posiadają one żadnych relacji w promieniu {hops}-hop."

        # 4. Format the result for the Agent
        output = [f"Otoczenie grafowe dla zapytania '{query}':"]
        for line in neighborhood_triples:
            output.append(f"- {line}")

        return "\n".join(output)

    def get_node_details(self, node_id: str) -> str:
        """
        [Action: inspect_node_details]
        Returns the full properties of a node by ID.
        """
        query = "MATCH (n) WHERE elementId(n) = $id RETURN properties(n) as props"
        with self.driver.session() as session:
            res = session.run(query, id=node_id).single()
            if res:
                props = res['props']
                # Remove the embedding from the preview, as it is unreadable for the LLM
                if 'embedding' in props:
                    del props['embedding']
                return json.dumps(props, indent=2, ensure_ascii=False)
            return f"Node with ID {node_id} not found."

    def merge_nodes(self, node_ids: List[str]) -> str:
        """
        [Action: merge_nodes]
        Merges nodes into one. The first ID on the list is the 'main' node the rest are merged into.
        """
        if len(node_ids) < 2:
            return "Error: Need at least 2 nodes to merge."

        query = """
        MATCH (n:Node) WHERE elementId(n) IN $ids
        WITH collect(n) as nodes
        CALL apoc.refactor.mergeNodes(nodes, {
            properties: "discard",
            mergeRels: true
        }) YIELD node
        RETURN elementId(node) as new_id, node.name as name
        """
        try:
            with self.driver.session() as session:
                res = session.run(query, ids=node_ids).single()
                return f"Successfully merged nodes into {res['name']} (ID: {res['new_id']})."
        except Exception as e:
            return f"Error merging nodes: {str(e)}"

    def deduplicate_relations_by_ids(self, relation_ids: List[str]) -> str:
        """
        [Action: deduplicate_relations]
        Merges the given relations (by ID) into one.
        Used when the agent sees e.g. 3 'KNOWS' relations between the same people.
        """
        if len(relation_ids) < 2:
            return "Error: Need at least 2 relations to merge."

        query = """
        MATCH ()-[r]->() WHERE elementId(r) IN $ids
        WITH collect(r) as rels
        CALL apoc.refactor.mergeRelationships(rels, {properties: "overwrite"})
        YIELD rel
        RETURN elementId(rel) as new_id, type(rel) as type
        """
        try:
            with self.driver.session() as session:
                res = session.run(query, ids=relation_ids).single()
                return f"Merged {len(relation_ids)} relations into one {res['type']} (ID: {res['new_id']})."
        except Exception as e:
            return f"Error merging relations: {str(e)}"

    def upsert_triplet(self, head: str, head_type: str, relation: str, tail: str, tail_type: str, chunk_id: Optional[str] = "manual_fix") -> str:
        """
        [Action: add_triplet]
        Adds or updates a triple.
        Generates embeddings on the fly to keep vector search consistent.
        """
        try:
            # Generate embeddings
            head_emb = self._embed_node(head, head_type)
            tail_emb = self._embed_node(tail, tail_type)
            rel_emb = self._embed_text(relation)

            rel_obj = Relation(head=head, head_type=head_type, relation=relation, tail=tail, tail_type=tail_type)

            with self.driver.session() as session:
                session.execute_write(
                    self._merge_relation,
                    chunk_id,
                    rel_obj,
                    head_emb,
                    tail_emb,
                    rel_emb
                )
            return f"Upserted triplet: ({head}) -[{relation}]-> ({tail})"
        except Exception as e:
            return f"Error adding triplet: {str(e)}"

    def update_node_properties(self, node_id: str, new_name: str = None, new_type: str = None) -> str:
        """
        [Action: update_node_properties]
        Updates the name or type of a node.
        Important: After a name/type change it also updates the node embedding!
        """
        try:
            with self.driver.session() as session:
                # 1. Fetch the current data if any parameter is None
                curr = session.run("MATCH (n) WHERE elementId(n) = $id RETURN n.name as name, n.type as type", id=node_id).single()
                if not curr:
                    return f"Node {node_id} not found."

                final_name = new_name if new_name else curr['name']
                final_type = new_type if new_type else curr['type']

                # 2. Recompute the embedding
                new_emb = self._embed_node(final_name, final_type)
                new_text = f"{final_type}: {final_name}"

                # 3. Update in the database
                query = """
                MATCH (n) WHERE elementId(n) = $id
                SET n.name = $name,
                    n.type = $type,
                    n.text = $text,
                    n.embedding = $emb
                RETURN n.name
                """
                session.run(query, id=node_id, name=final_name, type=final_type, text=new_text, emb=new_emb)

            return f"Updated node {node_id}. New state: {final_name} ({final_type})"
        except Exception as e:
            return f"Error updating node: {str(e)}"

    def split_node(self, original_node_id: str, new_name: str, new_type: str, relation_ids_to_move: List[str]) -> str:
        """
        [Action: split_node_and_move_relations]
        Creates a new node and moves the given relations to it from the old node.
        """
        try:
            # Recompute the embedding for the new node
            new_emb = self._embed_node(new_name, new_type)
            new_text = f"{new_type}: {new_name}"

            query = """
            MATCH (old) WHERE elementId(old) = $old_id

            // Create new node
            CREATE (new_node:Node {
                name: $new_name,
                type: $new_type,
                text: $new_text,
                embedding: $new_emb,
                created_from_split: $old_id
            })

            // Move relations
            WITH old, new_node
            MATCH (old)-[r]-(neighbor)
            WHERE elementId(r) IN $rel_ids
            CALL apoc.refactor.to(r, new_node) YIELD input, output

            RETURN count(input) as moved_count
            """
            with self.driver.session() as session:
                res = session.run(query,
                                  old_id=original_node_id,
                                  new_name=new_name,
                                  new_type=new_type,
                                  new_text=new_text,
                                  new_emb=new_emb,
                                  rel_ids=relation_ids_to_move).single()

            return f"Split node {original_node_id}. Created '{new_name}' and moved {res['moved_count']} relations."
        except Exception as e:
            return f"Error splitting node: {str(e)}"

    def redirect_relation(self, relation_id: str, new_target_node_id: str) -> str:
        """
        [Action: redirect_relation]
        Changes the end node of a relation (End Node) to another one.
        Useful when a relation points to the wrong person/object.
        """
        query = """
        MATCH ()-[r]->(), (new_target)
        WHERE elementId(r) = $rid AND elementId(new_target) = $ntid
        WITH r, new_target
        CALL apoc.refactor.to(r, new_target) YIELD input, output
        RETURN startNode(output).name as s, type(output) as rel_name, endNode(output).name as t
        """
        try:
            with self.driver.session() as session:
                res = session.run(query, rid=relation_id, ntid=new_target_node_id).single()
                if res:
                    return f"Redirected relation to point to: {res['t']}"
                return "Could not redirect relation (ids might be wrong)."
        except Exception as e:
            return f"Error redirecting relation: {str(e)}"

    def delete_element(self, element_id: str, is_node: bool = True) -> str:
        """
        [Action: delete_element]
        Deletes a node (with its relations) or a single relation.
        """
        try:
            with self.driver.session() as session:
                if is_node:
                    # DETACH DELETE removes the node and all relations attached to it
                    query = "MATCH (n) WHERE elementId(n) = $id DETACH DELETE n"
                    msg = f"Deleted NODE {element_id} and its relationships."
                else:
                    query = "MATCH ()-[r]->() WHERE elementId(r) = $id DELETE r"
                    msg = f"Deleted RELATION {element_id}."

                session.run(query, id=element_id)
                return msg
        except Exception as e:
            return f"Error deleting element: {str(e)}"

graph_repo = GraphRepository()

# Tests of the graph repo methods for agent actions

- ~~read_graph_schema~~
- ~~search_neighborhood~~
- ~~inspect_node_details~~
- ~~merge_nodes~~
- ~~deduplicate_relations~~
- ~~add_triplet~~
- ~~update_node_properties~~
- ~~split_node_and_move_relations~~  (ok but leaves an orphaned node behind)
- ~~redirect_relation~~
- ~~delete_element~~
- ~~do_nothing~~

In [ ]:
print("--- TEST: read_graph_schema ---")
schema = graph_repo.get_schema_summary()
print(schema)

print("\n--- TEST: inspect_node_details ---")
# First fetch any node ID from the database for the test
with graph_repo.driver.session() as s:
    node_id = s.run("MATCH (n:Node) RETURN elementId(n) LIMIT 1").single()
    if node_id:
        details = graph_repo.get_node_details(node_id[0])
        print(f"Details of node {node_id[0]}:\n{details}")
    else:
        print("The database is empty, no node to inspect.")

In [ ]:
print("--- TEST: search_neighborhood ---")

context_view = graph_repo.search_neighborhood(query="Warszawa", top_k=3, hops=1)

print(f"Agent view for 'Warszawa':")
print("-" * 30)
print(context_view)
print("-" * 30)

In [ ]:
print("--- TEST: add_triplet ---")
res = graph_repo.upsert_triplet(
    head="Władimir Putin",
    head_type="Person",
    relation="PRESIDENT_OF",
    tail="Rosja",
    tail_type="Country",
    chunk_id="test_chunk_001"
)
print(res)

# Add a second, similar node for the merge tests
graph_repo.upsert_triplet(
    head="Putin",
    head_type="Person",
    relation="LIVES_IN",
    tail="Moskwa",
    tail_type="City"
)

In [ ]:
print("--- TEST: merge_nodes ---")
with graph_repo.driver.session() as s:
    nodes = s.run("MATCH (n:Node) WHERE n.name IN ['Władimir Putin', 'Putin'] RETURN elementId(n) as id").data()
    ids = [r['id'] for r in nodes]

    if len(ids) >= 2:
        # The first on the list becomes the main one
        merge_res = graph_repo.merge_nodes(ids)
        print(merge_res)
    else:
        print("Not enough nodes found to merge.")

In [ ]:
print("--- TEST: update_node_properties ---")
with graph_repo.driver.session() as s:
    target = s.run("MATCH (n:Node {name: 'Władimir Putin'}) RETURN elementId(n) as id").single()
    if target:
        res = graph_repo.update_node_properties(
            node_id=target['id'],
            new_name="Vladimir V. Putin",
            new_type="Politician"
        )
        print(res)

In [ ]:
print("--- TEST: split_node ---")
# Scenario: We have a node "Zamek" that has a relation to "Warszawa" and "Malbork".
# We want to split out "Zamek w Malborku" as a separate entity.

graph_repo.upsert_triplet("Zamek", "Building", "LOCATED_IN", "Warszawa", "City")
graph_repo.upsert_triplet("Zamek", "Building", "LOCATED_IN", "Malbork", "City")

with graph_repo.driver.session() as s:
    # Find the node ID and the ID of the relation leading to Malbork
    data = s.run("""
        MATCH (n:Node {name: 'Zamek'})-[r:RELATED]->(t:Node {name: 'Malbork'})
        RETURN elementId(n) as nid, elementId(r) as rid
    """).single()

    if data:
        res = graph_repo.split_node(
            original_node_id=data['nid'],
            new_name="Zamek w Malborku",
            new_type="Building",
            relation_ids_to_move=[data['rid']]
        )
        print(res)

In [ ]:
print("--- TEST: redirect_relation ---")
# Scenario: The "CAPITAL" relation leads from Warsaw to Russia (error). We redirect it to Poland.

graph_repo.upsert_triplet("Warszawa", "City", "CAPITAL_OF", "Rosja", "Country")
graph_repo.upsert_triplet("Polska", "Country", "EXIST", "True", "Boolean")

with graph_repo.driver.session() as s:
    rel_data = s.run("""
        MATCH (c:Node {name: 'Warszawa', type: 'City'})-[r:RELATED]->(err:Node {name: 'Rosja'})
        MATCH (correct:Node {name: 'Polska'})
        RETURN elementId(r) as rid, elementId(correct) as cid
        """).single()

    if rel_data:
        res = graph_repo.redirect_relation(rel_data['rid'], rel_data['cid'])
        print(res)

In [ ]:
print("--- TEST: delete_element ---")
with graph_repo.driver.session() as s:
    # Delete the test node "True"
    target = s.run("MATCH (n:Node {name: 'True'}) RETURN elementId(n) as id").single()
    if target:
        res = graph_repo.delete_element(target['id'], is_node=True)
        print(res)

In [ ]:
print("--- TEST: deduplicate_relations_by_ids (Forced) ---")

with graph_repo.driver.session() as s:
    s.run("""
        MERGE (a:Node {name: 'Testowy User', type: 'Person'})
        MERGE (b:Node {name: 'Testowa Firma', type: 'Company'})
        CREATE (a)-[r1:RELATED {type: 'WORKS_AT', source: 'doc_1'}]->(b)
        CREATE (a)-[r2:RELATED {type: 'WORKS_AT', source: 'doc_2'}]->(b)
    """)

    rel_data = s.run("""
        MATCH (h:Node {name: 'Testowy User'})-[r:RELATED]->(t:Node {name: 'Testowa Firma'})
        RETURN elementId(r) as rid
    """).data()

    ids = [record['rid'] for record in rel_data]

    if len(ids) >= 2:
        print(f"Success! Found {len(ids)} relations. Running deduplication...")

        res = graph_repo.deduplicate_relations_by_ids(ids)
        print(res)

        final_count = s.run("""
            MATCH (h:Node {name: 'Testowy User'})-[r:RELATED]->(t:Node {name: 'Testowa Firma'})
            RETURN count(r) as c
        """).single()['c']
        print(f"Number of relations after the operation: {final_count}")
    else:
        print("Still no duplicates found.")

# Agent

In [ ]:
from typing import Annotated, List, TypedDict, Union, Dict, Any
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.tools import tool
from typing import List, Optional
from langchain_ollama import ChatOllama
import operator

# --- State Definition ---
class AgentState(TypedDict):
    input_subgraph: List[Dict]  # The new data to integrate
    graph_context: str  # The current view of the graph
    messages: Annotated[List[BaseMessage], operator.add] # Chat history
    schema_summary: str  # Global list of node/rel types
    iterations: int  # Safety counter

In [ ]:
@tool
def read_graph_schema_tool():
    """
    [Action: read_graph_schema]
    Retrieves a summary of the graph schema, including unique node types (labels) and relationship types.
    Use this to understand the ontology and vocabulary of the graph.
    """
    return graph_repo.get_schema_summary()

@tool
def search_neighborhood_tool(query: str):
    """
    [Action: search_neighborhood]
    Searches for nodes matching the query (using vector/keyword search) and returns their immediate neighborhood (1-hop).
    Search for a SINGLE entity name to see its local graph. Do not provide full sentences.
    Use this to visualize the graph context around a specific concept or entity before making changes.
    """
    return graph_repo.search_neighborhood(query, top_k=3, hops=1)

@tool
def inspect_node_details_tool(node_id: str):
    """
    [Action: inspect_node_details]
    Returns all properties of a specific node given its elementId.
    Use this to verify details (e.g., dates, URLs) before merging or deleting.
    """
    return graph_repo.get_node_details(node_id)

@tool
def merge_nodes_tool(node_ids: List[str]):
    """
    [Action: merge_nodes]
    Merges multiple nodes into a single node. The first ID in the list becomes the primary node; others are merged into it.
    Relationships are preserved and moved to the primary node.
    Use this for deduplication (e.g., merging "USA" and "United States").
    """
    return graph_repo.merge_nodes(node_ids)

@tool
def deduplicate_relations_tool(relation_ids: List[str]):
    """
    [Action: deduplicate_relations]
    Merges multiple relationships (specified by their IDs) into a single relationship.
    Use this when you see multiple identical edges (e.g., three 'KNOWS' edges) between the same two nodes.
    """
    return graph_repo.deduplicate_relations_by_ids(relation_ids)

@tool
def add_triplet_tool(head: str, head_type: str, relation: str, tail: str, tail_type: str):
    """
    [Action: add_triplet]
    Adds a new fact (triplet) to the graph: (Head)-[Relation]->(Tail).
    If nodes do not exist, they are created. If they exist, the relationship is added/merged.
    Use this to insert new knowledge from the input chunk.
    """
    return graph_repo.upsert_triplet(head, head_type, relation, tail, tail_type)

@tool
def update_node_properties_tool(node_id: str, new_name: str = None, new_type: str = None):
    """
    [Action: update_node_properties]
    Updates a node's name or type. Automatically refreshes the node's embedding.
    Use this to correct typos or normalize names (e.g., changing "Jan" to "Jan Kowalski").
    """
    return graph_repo.update_node_properties(node_id, new_name, new_type)

@tool
def split_node_tool(original_node_id: str, new_name: str, new_type: str, relation_ids_to_move: List[str]):
    """
    [Action: split_node_and_move_relations]
    Splits a 'super-node' that incorrectly aggregates distinct entities.
    Creates a new node with 'new_name' and moves the specified relationships (by ID) from the old node to the new one.
    """
    return graph_repo.split_node(original_node_id, new_name, new_type, relation_ids_to_move)

@tool
def redirect_relation_tool(relation_id: str, new_target_node_id: str):
    """
    [Action: redirect_relation]
    Redirects an existing relationship to point to a different target node.
    Use this when a relationship points to the wrong entity (e.g., 'Capital' pointing to 'Russia' instead of 'Moscow').
    """
    return graph_repo.redirect_relation(relation_id, new_target_node_id)

@tool
def delete_element_tool(element_id: str, is_node: bool = True):
    """
    [Action: delete_element]
    Deletes a node (and its relationships) OR a single relationship.
    Set is_node=True to delete a node. Set is_node=False to delete a relationship.
    """
    return graph_repo.delete_element(element_id, is_node)

@tool
def do_nothing_tool():
    """
    [Action: do_nothing]
    Signals that the graph is consistent with the input data and no further actions are needed.
    Call this to finish the process.
    """
    return "Process finished."

# List of tools for the agent
tools = [
    read_graph_schema_tool,
    search_neighborhood_tool,
    inspect_node_details_tool,
    merge_nodes_tool,
    deduplicate_relations_tool,
    add_triplet_tool,
    update_node_properties_tool,
    split_node_tool,
    redirect_relation_tool,
    delete_element_tool,
    do_nothing_tool
]

In [ ]:
def retrieve_context_node(state: AgentState):
    """
    Refreshes the 'graph_context' based on the entities in 'input_subgraph'.
    This runs at the start AND after every tool execution to show the agent the new state.
    """
    input_data = state["input_subgraph"]
    schema = state.get("schema_summary", "")

    if not schema:
        schema = graph_repo.get_schema_summary()

    # Extract entities to search for
    nodes_to_search = []
    for rel in input_data:
        nodes_to_search.append({"name": rel["head"], "type": rel["head_type"]})
        nodes_to_search.append({"name": rel["tail"], "type": rel["tail_type"]})

    # 1. Find nearest nodes (Vector Search)
    nearest_nodes = graph_repo.find_nearest_nodes(nodes_to_search, top_k=3)

    # 2. Extract IDs for expansion
    ids_to_expand = set()
    for node_list in nearest_nodes.values():
        for node in node_list:
            if 'id' in node:
                ids_to_expand.add(node['id'])

    # 3. Expand Subgraph (Traverse 1-hop) - This returns strings with IDs!
    # Using the repository method directly to get the list of strings
    context_lines = graph_repo._expand_subgraph(list(ids_to_expand), hops=1, limit=30)

    if context_lines:
        context_str = "CURRENT GRAPH STATE (NEO4J):\n" + "\n".join(context_lines)
    else:
        context_str = "CURRENT GRAPH STATE: No relevant existing nodes found (Graph might be empty)."

    return {"graph_context": context_str, "schema_summary": schema}

In [ ]:
def curator_agent_node(state: AgentState):
    """
    The reasoning engine. Decides what to do based on Input vs. Context.
    """
    iteration = state.get("iterations", 0)
    input_data = state["input_subgraph"]
    context = state["graph_context"]
    schema = state.get("schema_summary", "Schema not available.")

    model = ChatOllama(model=LLM_MODEL_NAME, temperature=0, reasoning=True, num_ctx=131000, num_predict=-2)
    model_with_tools = model.bind_tools(tools)

    # Chain of Thought prompt in English with Polish data requirements
    system_msg = f"""You are a Knowledge Graph Curator Expert.
    Your goal is to maintain absolute consistency and cleanliness of the Neo4j database while integrating new knowledge.

    LANGUAGE REQUIREMENT:
    - All node names (name), types (type), and relationship names (relation/type) MUST be in POLISH.
    - If you encounter English terms in the Input, translate them to Polish before adding or updating.
    - Consistency in Polish terminology is critical (e.g., use "Osoba" instead of "Person").

    ONTOLOGY & SCHEMA COMPLIANCE (Guideline):
    You have access to the "GLOBAL SCHEMA" below.
    1. PRIORITIZE EXISTING TYPES: Always check the Global Schema first. If a suitable Node Type or Relationship Type exists there, USE IT.
       - Example: If the schema contains "Osoba", do NOT create a new type "Ludzie" or "Person".
    2. NEW TYPES ALLOWED: If the Input introduces a completely new concept that cannot be mapped to existing types in the Schema, you ARE ALLOWED to create a new type.
    3. NAMING CONVENTION: When creating new types, use Singular, PascalCase, and Polish language (e.g., "StatekKosmiczny", "Umowa").

    ERROR HANDLING & RECOVERY:
    If a tool execution fails (returns an error message), you must self-correct:
    - "Node not found": The ID might be wrong or the node was merged/deleted. Use `search_neighborhood_tool` to find the correct entity ID again.
    - "Merge failed": Check if you are trying to merge nodes that are already merged. Refresh context.
    - "Recursion Limit": If you feel stuck in a loop, STOP. If the graph is mostly correct, use `do_nothing_tool`.
    - Infinite Adding: If you see the same triple in 'Input' and 'Current Graph', DO NOT add it again.

    YOUR TASK:
    You have a list of new facts (INPUT SUBGRAPH) and the current state of the database (CURRENT GRAPH CONTEXT).
    You must integrate the Input into the Context, resolving conflicts and duplicates.

    MANDATORY OPERATING PROTOCOL (Step-by-Step):

    PHASE 1: DIAGNOSIS AND CLEANING (Highest Priority)
    Before adding anything new, fix what you already see in the Context.
    1. NODE DUPLICATES:
       - Do you see "Władimir Putin" (ID: 123) and "Putin" (ID: 456) in the context?
       - If it's the same entity -> USE `merge_nodes_tool([123, 456])`.
       - IMPORTANT: Do not merge nodes of different types (e.g., City and Person) unless it's an obvious error.

    2. RELATIONSHIP DUPLICATES:
       - Do you see multiple arrows of the same type between the same nodes?
       - USE `deduplicate_relations_tool`.

    3. INCORRECT CONNECTIONS:
       - Does a relationship point to the wrong node (e.g., "Stolica" -> "Rosja" instead of "Moskwa")?
       - USE `redirect_relation_tool`.

    4. RESOLVING AMBIGUITY (SPLITTING):
       - Does a node "Zamek" connect to both "Warszawa" and "Malbork"? This is an error.
       - USE `split_node_tool` to separate "Zamek w Malborku".

    PHASE 2: INTEGRATION (Only when Context is clean)
    1. If the Input contains facts not present in the Context -> USE `add_triplet_tool`.
    2. If the Input confirms existing relationships -> DO NOTHING (avoid redundancy).

    PHASE 3: METADATA UPDATES
    - If a node is named "Jan" and Input says "Jan Kowalski" -> USE `update_node_properties_tool`.

    CRITICAL RULES:
    - WORK ITERATIVELY. Perform 1-2 corrective actions, then wait for the context to refresh.
    - DO NOT GUESS IDs. Use only IDs visible in the CURRENT GRAPH CONTEXT section.
    - FINISH FAST: If the INPUT SUBGRAPH is already reflected in the CURRENT GRAPH CONTEXT, you MUST use do_nothing_tool immediately.
    - DO NOT UNDO YOUR WORK: Do not delete or redirect relations you have just created in previous steps unless you explicitly identified a mistake in your reasoning.
    - AVOID SEARCH LOOPS: If search_neighborhood_tool returns the same information twice, stop searching and decide based on what you have.
    - If the Context is empty, simply add the triplets from the Input.
    - If the work is done and the Input is fully reflected in the graph -> USE `do_nothing_tool`.

    --- INPUT DATA ---
    INPUT SUBGRAPH (New knowledge):
    {json.dumps(input_data, indent=2, ensure_ascii=False)}

    --- GLOBAL SCHEMA (Available Types) ---
    {schema}

    --- CURRENT DATABASE STATE ---
    CURRENT GRAPH CONTEXT (As seen in the DB):
    {context}

    Iteration: {iteration}/10"""

    response = model_with_tools.invoke([SystemMessage(content=system_msg)] + state["messages"])

    return {
        "messages": [response],
        "iterations": iteration + 1
    }

In [ ]:
workflow = StateGraph(AgentState)

# Nodes
workflow.add_node("retrieve_context", retrieve_context_node)
workflow.add_node("curator", curator_agent_node)
workflow.add_node("tools", ToolNode(tools))

# Edges
workflow.add_edge(START, "retrieve_context")
workflow.add_edge("retrieve_context", "curator")

# Router
def router(state: AgentState):
    iteration = state.get("iterations", 0)
    last_message = state["messages"][-1]

    if iteration > 10:
        return END

    if last_message.tool_calls:
        # Check if the agent wants to stop
        if last_message.tool_calls[0]["name"] == "do_nothing_tool":
            return END
        return "tools"

    return END


workflow.add_conditional_edges("curator", router, {"tools": "tools", END: END})
workflow.add_edge("tools", "retrieve_context") # Loop back to refresh context

app = workflow.compile()

In [ ]:
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
input_chunk = [
    {"head": "Elon Musk", "head_type": "Person", "relation": "CEO_OF", "tail": "Tesla", "tail_type": "Company"},
    {"head": "Tesla", "head_type": "Company", "relation": "HQ_IN", "tail": "Texas", "tail_type": "Location"}
]

initial_state = {
    "input_subgraph": input_chunk,
    "graph_context": "",
    "messages": [],
    "iterations": 0
}

print("--- Starting Agent ---")
try:
    for event in app.stream(initial_state, {"recursion_limit": 50}):
        for key, value in event.items():
            print(f"\n[Finished Node: {key}]")
            if key == "curator":
                msg = value["messages"][-1]
                if msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"Tool Call: {tc['name']} args: {tc['args']}")
                else:
                    print("Agent Response:", msg.content)
            elif key == "retrieve_context":
                # Printing context to debug what the agent sees
                print(value['graph_context'])
                pass
except Exception as e:
    print(f"Workflow Error: {e}")

# TODO:
  - ~~The agent should ALWAYS get the list of available relations and nodes from the main graph at the start and be able to call it along the way (fix prompts and workflow)~~
  - ~~Fix the node and relation creation prompt so there are no English names and types~~
  - ~~The agent should be able to handle errors if they occurred (fix prompts and workflow)~~
  - ~~add an action for the agent to preview the source chunk (e.g. node id as parameter, output is the chunk text)~~
 - ~~parallel tool calls (to speed up chunk processing and let the agent push several Relations at once)~~
 - ~~fix 'dangling', standalone nodes - maybe they should not be in the graph (e.g. 'Wielki błąd')~~
 - ~~ids from neo4j (of nodes and relations) passed to the agent context are too long - they should be shortened e.g. by mapping 'on the fly'~~
